In [ ]:
from ultralytics import YOLO
import os

In [ ]:
from pathlib import Path

print("Train images :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\images\train").glob("*.png"))))
print("Train labels :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\labels\train").glob("*.txt"))))

print("Val images :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\images\val").glob("*.png"))))
print("Val labels :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\labels\val").glob("*.txt"))))

In [ ]:
from ultralytics.data.utils import check_det_dataset

check_det_dataset(
    r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\acdc_rain.yaml"
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")

print("ACDC Rain Training Started ")

model.train(
    data=r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\acdc_rain.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    workers=4,
    device=0,
    cache=False,
    amp=True,
    project="ACDC_RAIN_Project",
    name="train_yolov8l_rain"
)

print(" Training Finished ")

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

# Configuration
results_dir = r'C:\Users\Varis\runs\detect\ACDC_Rain_Project\train_yolov8l_rain'
results_csv = os.path.join(results_dir, 'results.csv')
output_report = os.path.join(results_dir, 'training_summary_report.txt')

def generate_summary():
    if not os.path.exists(results_csv):
        print("Results CSV not found.")
        return

    data = pd.read_csv(results_csv)
    data.columns = data.columns.str.strip()

    # Generate Graphs
    fig, ax = plt.subplots(2, 2, figsize=(12, 10))
    
    ax[0, 0].plot(data['epoch'], data['train/box_loss'], label='Box Loss')
    ax[0, 0].set_title('Box Loss')
    
    ax[0, 1].plot(data['epoch'], data['metrics/mAP50(B)'], color='orange', label='mAP@50')
    ax[0, 1].set_title('mAP@50')
    
    ax[1, 0].plot(data['epoch'], data['metrics/mAP50-95(B)'], color='green', label='mAP@50-95')
    ax[1, 0].set_title('mAP@50-95')
    
    ax[1, 1].plot(data['epoch'], data['metrics/precision(B)'], color='red', label='Precision')
    ax[1, 1].set_title('Precision')

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'performance_graphs.png'))

    # Generate Text Report
    best_row = data.loc[data['metrics/mAP50-95(B)'].idxmax()]
    
    with open(output_report, 'w') as f:
        f.write("Training Summary Report\n")
        f.write("========================\n")
        f.write(f"Total Epochs: 100\n")
        f.write(f"Best Epoch: {int(best_row['epoch'])}\n")
        f.write(f"Best mAP@50: {best_row['metrics/mAP50(B)']:.4f}\n")
        f.write(f"Best mAP@50-95: {best_row['metrics/mAP50-95(B)']:.4f}\n")
        f.write(f"Final Box Loss: {data.iloc[-1]['train/box_loss']:.4f}\n")
        f.write("\nProcess completed successfully.")

    print(f"Report and Graphs saved in {results_dir}")

if __name__ == "__main__":
    generate_summary()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import shutil

# Paths
source_dir = r'C:\Users\Varis\runs\detect\ACDC_Rain_Project\train_yolov8l_rain'
backup_dir = r'D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_RAIN\Training_Backup'

# Create backup directory if it doesn't exist
os.makedirs(backup_dir, exist_ok=True)

def generate_and_copy():
    results_csv = os.path.join(source_dir, 'results.csv')
    
    if not os.path.exists(results_csv):
        print(f"Error: {results_csv} not found.")
        return

    data = pd.read_csv(results_csv)
    data.columns = data.columns.str.strip()

    # 1. Generate Performance Graphs
    fig, ax = plt.subplots(2, 2, figsize=(12, 10))
    ax[0, 0].plot(data['epoch'], data['train/box_loss'])
    ax[0, 0].set_title('Box Loss')
    ax[0, 1].plot(data['epoch'], data['metrics/mAP50(B)'], color='orange')
    ax[0, 1].set_title('mAP@50')
    ax[1, 0].plot(data['epoch'], data['metrics/mAP50-95(B)'], color='green')
    ax[1, 0].set_title('mAP@50-95')
    ax[1, 1].plot(data['epoch'], data['metrics/precision(B)'], color='red')
    ax[1, 1].set_title('Precision')
    
    plt.tight_layout()
    plt.savefig(os.path.join(backup_dir, 'performance_graphs.png'))
    plt.close()

    # 2. Generate Text Report
    best_row = data.loc[data['metrics/mAP50-95(B)'].idxmax()]
    report_path = os.path.join(backup_dir, 'training_summary.txt')
    
    with open(report_path, 'w') as f:
        f.write("YOLOv8 Training Summary - ACDC Rain\n")
        f.write("------------------------------------\n")
        f.write(f"Total Epochs: 100\n")
        f.write(f"Best Epoch: {int(best_row['epoch'])}\n")
        f.write(f"Best mAP@50: {best_row['metrics/mAP50(B)']:.4f}\n")
        f.write(f"Best mAP@50-95: {best_row['metrics/mAP50-95(B)']:.4f}\n")
        f.write(f"Final Precision: {data.iloc[-1]['metrics/precision(B)']:.4f}\n")
        f.write(f"Final Recall: {data.iloc[-1]['metrics/recall(B)']:.4f}\n")

    # 3. Copy Key Files
    files_to_copy = ['best.pt', 'last.pt', 'confusion_matrix.png', 'results.csv']
    for file in files_to_copy:
        src = os.path.join(source_dir, 'weights' if 'pt' in file else '', file)
        if os.path.exists(src):
            shutil.copy(src, backup_dir)
            
    print(f"Backup completed successfully in: {backup_dir}")

if __name__ == "__main__":
    generate_and_copy()